# 01 — De-risk prototype: is a within-combine *fatigue* signal identifiable?

**Project:** NFL Big Data Bowl 2027 — combine-fatigue study (`conormalone/BDB27`).
**Scope:** cheap reconnaissance, *not* the full study. Three questions:

1. **Order / time-of-day recoverable?** Reconstruct each player's within-session drill sequence
   from `event_id` + `time`; check that `attempt` order agrees with `time`.
2. **Crude attempt-to-attempt decay** in `FORTY_YARD_DASH`, `THREE_CONE_DRILL`, `SHORT_SHUTTLE`.
3. **Confounding:** how badly is drill *order* collinear with (a) drill type and (b) position group —
   i.e. is a fatigue effect even identifiable?

**Data:** `combine_tracking.csv` (10 Hz, 463,190 rows) + `combine_results.csv`, read from `../data/raw/`
(symlink to the USB). Runs on a Raspberry Pi 5 / 4 GB — the tracking file is **streamed in chunks**;
the big `game_tracking_*` / `player_play.csv` files are never opened here.


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# --- paths (this notebook lives in BDB27/notebooks/) ---
RAW = Path("../data/raw")
TRACK = RAW / "combine_tracking.csv"
RESULTS = RAW / "combine_results.csv"
FIGDIR = Path("../outputs/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.30, "axes.spines.top": False,
    "axes.spines.right": False,
})

print("tracking file exists:", TRACK.exists())
print("results file exists :", RESULTS.exists())


## 1. Load: one row per *event* (attempt), memory-safe

Each `event_id` is exactly one attempt by one player in one drill (verified below). We stream the
tracking file in chunks, keep only `entity_type == "PLAYER"` rows, and aggregate per `event_id`.


In [ ]:
def load_event_summary(path, chunksize: int = 200_000) -> pd.DataFrame:
    """Stream the 10 Hz tracking file and return ONE ROW PER event_id.

    Memory-safe for the Pi: the 83 MB / 463k-row file is never held in memory at once.
    Only PLAYER rows are used for kinematics (BALL rows are dropped).
    """
    cols = ["draft_year", "event_id", "nfl_id", "entity_type", "time", "drill_type",
            "drill_name", "attempt", "x", "y", "s"]
    dtypes = {"event_id": "object", "nfl_id": "int64", "entity_type": "object",
              "time": "object", "drill_type": "object", "drill_name": "object",
              "draft_year": "int32", "x": "float64", "y": "float64", "s": "float64",
              "attempt": "int16"}
    acc: dict[str, dict] = {}
    for ch in pd.read_csv(path, usecols=cols, dtype=dtypes, chunksize=chunksize):
        ch = ch[ch.entity_type == "PLAYER"]
        if ch.empty:
            continue
        ch = ch.sort_values(["event_id", "time"], kind="stable")
        step = np.hypot(ch.groupby("event_id")["x"].diff(),
                        ch.groupby("event_id")["y"].diff())
        ch = ch.assign(_step=step)
        for eid, sub in ch.groupby("event_id", sort=False):
            st = sub._step.to_numpy()
            st = st[np.isfinite(st)]
            rec = dict(
                nfl_id=int(sub.nfl_id.iloc[0]), draft_year=int(sub.draft_year.iloc[0]),
                drill_type=sub.drill_type.iloc[0], drill_name=sub.drill_name.iloc[0],
                attempt=int(sub.attempt.iloc[0]),
                t_start=sub.time.iloc[0], t_end=sub.time.iloc[-1], n_frames=int(len(sub)),
                path_len=float(st.sum()), peak_speed=float(sub.s.max()),
                start_speed=float(sub.s.iloc[0]),
                x0=float(sub.x.iloc[0]), y0=float(sub.y.iloc[0]),
                x1=float(sub.x.iloc[-1]), y1=float(sub.y.iloc[-1]),
            )
            if eid in acc:  # event split across a chunk boundary -> merge
                o = acc[eid]
                o["t_end"] = max(o["t_end"], rec["t_end"])
                o["n_frames"] += rec["n_frames"]
                o["path_len"] += rec["path_len"]
                o["peak_speed"] = max(o["peak_speed"], rec["peak_speed"])
                o["x1"], o["y1"] = rec["x1"], rec["y1"]
            else:
                acc[eid] = rec
    E = pd.DataFrame.from_dict(acc, orient="index")
    E.index.name = "event_id"
    E = E.reset_index()
    E["t_start"] = pd.to_datetime(E.t_start)
    E["t_end"] = pd.to_datetime(E.t_end)
    E["duration_s"] = (E.t_end - E.t_start).dt.total_seconds()
    E["net_disp"] = np.hypot(E.x1 - E.x0, E.y1 - E.y0)
    E["tod_hours"] = E.t_start.dt.hour + E.t_start.dt.minute / 60.0
    return E


E = load_event_summary(TRACK)
print("events (attempts) :", len(E))
print("players           :", E.nfl_id.nunique())
print("event window      :", E.t_start.min(), "->", E.t_end.max())
print("frames/event      :", E.n_frames.describe()[["min", "50%", "max"]].to_dict())


### Add position group + define the *session*

`combine_results.csv` gives `combine_position`. A **session** is one player's contiguous block of
activity. Naively splitting on the calendar date is unsafe: many workouts run **past midnight**
(e.g. 22:17 -> 00:12), which would wrongly fragment a single session. We instead split on a
**2-hour inactivity gap** within each player's timeline.


In [ ]:
res = (pd.read_csv(RESULTS, usecols=["nfl_id", "combine_position"],
                    dtype={"nfl_id": "int64"})
         .drop_duplicates("nfl_id"))
E = E.merge(res, on="nfl_id", how="left")
print("events missing position:", int(E.combine_position.isna().sum()))

E = E.sort_values(["nfl_id", "t_start"], kind="stable").reset_index(drop=True)
gap_s = E.groupby("nfl_id")["t_start"].diff().dt.total_seconds()
E["new_session"] = (gap_s.isna() | (gap_s > 7200)).astype(int)     # >2 h break = new session
E["session"] = (E.nfl_id.astype(str) + "#"
                + E.groupby("nfl_id")["new_session"].cumsum().astype(str))
E["sess_rank"] = E.groupby("session").cumcount()                  # 0 = first test of session
E["sess_size"] = E.groupby("session")["event_id"].transform("size")
E["order_frac"] = np.where(E.sess_size > 1, E.sess_rank / (E.sess_size - 1), 0.0)
E["combine_day"] = E.groupby("session")["t_start"].transform("min").dt.date.astype(str)

print("consecutive-event gap (s):", gap_s.describe()[["min", "50%", "max"]].round(1).to_dict())
print("sessions per player:", E.groupby("nfl_id")["session"].nunique().value_counts().sort_index().to_dict())
print("events per session :", E.groupby("session").size().describe()[["min", "50%", "max"]].to_dict())
span_h = ((E.groupby("session").t_end.max() - E.groupby("session").t_start.min())
          .dt.total_seconds() / 3600)
print("session wall-clock span (h):", span_h.describe()[["min", "50%", "max"]].round(2).to_dict())


## 2. Q1 — Is order / time-of-day recoverable?

`time` is a full ISO-8601 timestamp, so **time-of-day is exact**. We test whether the organisers'
own `attempt` counter is consistent with the timestamp order.


In [ ]:
E2 = E.sort_values(["session", "drill_name", "t_start"], kind="stable").copy()
d = E2.groupby(["session", "drill_name"])["attempt"].diff()
E2["_ok"] = d.fillna(0) >= 0
E2["_all_ok"] = E2.groupby(["session", "drill_name"])["_ok"].transform("min")
mono_name = E2.drop_duplicates(["session", "drill_name"])["_all_ok"].mean()

E3 = E.sort_values(["session", "drill_type", "t_start"], kind="stable").copy()
d3 = E3.groupby(["session", "drill_type"])["attempt"].diff()
E3["_ok"] = d3.fillna(0) >= 0
E3["_all_ok"] = E3.groupby(["session", "drill_type"])["_ok"].transform("min")
mono_type = E3.drop_duplicates(["session", "drill_type"])["_all_ok"].mean()

print("attempt monotonic with time within (session, drill_name): %.1f%%  (%d groups)"
      % (100 * mono_name, E2.groupby(["session", "drill_name"]).ngroups))
print("attempt monotonic with time within (session, drill_type): %.1f%%  (%d groups)"
      % (100 * mono_type, E3.groupby(["session", "drill_type"]).ngroups))


### What does the within-session order actually look like?

In [ ]:
order_tab = (E.groupby("drill_type")["sess_rank"]
             .agg(n="size", min="min", q25=lambda s: s.quantile(.25),
                  median="median", q75=lambda s: s.quantile(.75), max="max")
             .sort_values("median"))
print(order_tab.round(1).to_string())

print("\nCanonical order by position group (median within-session rank):")
seq = (E.groupby(["combine_position", "drill_type"])["sess_rank"].median()
       .reset_index().sort_values(["combine_position", "sess_rank"]))
for pos, sub in seq.groupby("combine_position"):
    print(" ", pos, "->", " > ".join(sub.drill_type.tolist()))

print("\ntime-of-day (clock hour; timestamps look like UTC) by drill type:")
print(E.groupby("drill_type")["tod_hours"].agg(["min", "median", "max"]).round(2)
      .sort_values("median").to_string())
print("\nclock-hour coverage:", E.t_start.dt.hour.value_counts().sort_index().to_dict())
print("-> no on-field testing between ~02:00 and ~18:00; sessions run evening into the night.")


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5))
order = order_tab.index.tolist()
data = [E.loc[E.drill_type == dd, "sess_rank"].values for dd in order]
colors = plt.cm.viridis(np.linspace(.15, .85, len(order)))
bp = ax.boxplot(data, orientation="horizontal", tick_labels=order,
                patch_artist=True, showfliers=False)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(.8)
rng = np.random.default_rng(0)
for i, dd in enumerate(order):
    y = E.loc[E.drill_type == dd, "sess_rank"].values
    ax.scatter(y, rng.normal(i + 1, .07, len(y)), s=7, color="k", alpha=.25, zorder=3)
ax.set_xlabel("position within player's session  (rank; 0 = first test of the session)")
ax.set_title("Q1/Q3 \u2014 Combine test order is a fixed protocol\n(each drill occupies a narrow slice of the session)")
plt.tight_layout()
plt.savefig(FIGDIR / "fig1_drill_order.png", bbox_inches="tight")
plt.show()
print("saved", FIGDIR / "fig1_drill_order.png")


**Q1 read:** `attempt` agrees with `time` **100%** within `(session, drill_name)` — the counter is a
faithful ordering. It only looks non-monotonic within `drill_type` because the `SKILL_DRILLS_*`
families pack many sub-drills (routes) that each restart `attempt` at 1. Within-session order is
therefore **fully recoverable**, and it is essentially the combine's **fixed test protocol**:
`FORTY_YARD_DASH` first, then the positional skill drills, then `THREE_CONE_DRILL` / `SHORT_SHUTTLE`.
Each player shows up as exactly **one** ~1.5 h session (calendar-date splitting would have wrongly cut
111 of them across midnight). Timestamps look like **UTC** (workouts land at 13:00-21:00 US-Eastern).


## 3. Q2 — Crude attempt-to-attempt decay

We derive per-attempt kinematics straight from the 10 Hz tracking: **window duration**,
**peak speed** and **path length**. Sanity check: peak speed should track the official 40-yard time.


In [ ]:
res2 = pd.read_csv(RESULTS, usecols=["nfl_id", "draft_year", "forty", "three_cone", "short_shuttle"])
for drill, col in [("FORTY_YARD_DASH", "forty"),
                   ("THREE_CONE_DRILL", "three_cone"),
                   ("SHORT_SHUTTLE", "short_shuttle")]:
    d1 = (E[(E.drill_type == drill) & (E.attempt == 1)]
          .merge(res2[["nfl_id", "draft_year", col]], on=["nfl_id", "draft_year"], how="left"))
    d1[col] = pd.to_numeric(d1[col], errors="coerce")
    d1 = d1.dropna(subset=[col])
    print("%-17s n=%3d   corr(duration, official)=%+.3f   corr(peak_speed, official)=%+.3f"
          % (drill, len(d1), d1[["duration_s", col]].corr().iloc[0, 1],
             d1[["peak_speed", col]].corr().iloc[0, 1]))


In [ ]:
from scipy import stats


def paired_attempt_decay(drill: str, metric: str):
    """Within player-year, compare attempt 1 vs attempt 2 on `metric` (paired)."""
    d = E[E.drill_type == drill]
    piv = d.pivot_table(index=["nfl_id", "draft_year"], columns="attempt",
                        values=metric, aggfunc="first")
    if 1 not in piv or 2 not in piv:
        return None
    sub = piv[[1, 2]].dropna()
    if len(sub) < 5:
        return dict(n=len(sub))
    delta = sub[2] - sub[1]
    _, p = stats.ttest_rel(sub[2], sub[1])
    return dict(n=len(sub), mean_att1=sub[1].mean(), mean_att2=sub[2].mean(),
                mean_delta=delta.mean(), p_value=p)


rows = []
for drill in ["FORTY_YARD_DASH", "THREE_CONE_DRILL", "SHORT_SHUTTLE"]:
    for metric in ["duration_s", "peak_speed", "path_len"]:
        r = paired_attempt_decay(drill, metric)
        if r:
            rows.append(dict(drill=drill, metric=metric, **r))
dec = pd.DataFrame(rows)
print(dec.round(4).to_string(index=False))


In [ ]:
for drill in ["FORTY_YARD_DASH", "THREE_CONE_DRILL", "SHORT_SHUTTLE"]:
    d = E[E.drill_type == drill]
    print("\n=== %s ===" % drill, "attempt counts:",
          d.attempt.value_counts().sort_index().to_dict())
    print(d.groupby("attempt").agg(n=("duration_s", "size"),
                                   duration_s=("duration_s", "mean"),
                                   peak_speed=("peak_speed", "mean"),
                                   path_len=("path_len", "mean")).round(3).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
d = E[E.drill_type == "FORTY_YARD_DASH"]
m = d.groupby("attempt")["peak_speed"].agg(["mean", "sem"])
axes[0].errorbar(m.index, m["mean"], yerr=m["sem"], marker="o", capsize=3, color="#1f3b73")
axes[0].set_title("FORTY_YARD_DASH: peak speed by attempt")
axes[0].set_xlabel("attempt number"); axes[0].set_ylabel("peak speed (yd/s)")
m2 = d.groupby("attempt")["duration_s"].agg(["mean", "sem"])
axes[1].errorbar(m2.index, m2["mean"], yerr=m2["sem"], marker="o", capsize=3, color="#8c1c1c")
axes[1].set_title("FORTY_YARD_DASH: window duration by attempt")
axes[1].set_xlabel("attempt number"); axes[1].set_ylabel("duration (s)")
plt.tight_layout()
plt.savefig(FIGDIR / "fig2_attempt_decay.png", bbox_inches="tight")
plt.show()
print("saved", FIGDIR / "fig2_attempt_decay.png")


**Q2 read:** the tracking-derived metrics are credible (peak speed correlates ~**-0.98** with the
official 40-yard time; duration ~**+0.81**). But the crude attempt-to-attempt signal is **tiny**:
40-yard peak speed drops only ~**0.02 yd/s** from attempt 1->2 (statistically detectable, p~0.001,
but ~0.2% of the mean), duration is flat, and 3-cone / shuttle show no consistent direction with
very small paired samples. There is **no sizeable within-attempt fatigue decrement** in this crude
pass — consistent with warm-up / potentiation offsetting fatigue on repeated attempts.


## 4. Q3 — Confounding: is order separable from drill type and position?

We quantify how much of the within-session **order** is explained by drill type / name / position
(eta-squared), and how entangled drill type and position group are (Cramer's V), plus the day structure.


In [ ]:
def eta_squared(values, groups) -> float:
    df = pd.DataFrame({"v": np.asarray(values, float), "g": np.asarray(groups)}).dropna()
    gm = df.v.mean()
    ss_tot = ((df.v - gm) ** 2).sum()
    gm_g = df.groupby("g")["v"].transform("mean")
    cnt = df.groupby("g")["v"].transform("size")
    ss_bet = (cnt * (gm_g - gm) ** 2).groupby(df.g).first().sum()
    return float(ss_bet / ss_tot)


def cramers_v(a, b) -> float:
    tab = pd.crosstab(a, b).values.astype(float)
    n = tab.sum()
    exp = np.outer(tab.sum(axis=1), tab.sum(axis=0)) / n
    chi2 = ((tab - exp) ** 2 / exp).sum()
    r, k = tab.shape
    return float(np.sqrt((chi2 / n) / (min(r - 1, k - 1))))


print("within-session ORDER ~ (eta^2):")
print("  drill_type : %.3f" % eta_squared(E.order_frac, E.drill_type))
print("  drill_name : %.3f" % eta_squared(E.order_frac, E.drill_name))
print("  position   : %.3f" % eta_squared(E.order_frac, E.combine_position))
print()
print("Cramer's V  drill_type x position : %.3f" % cramers_v(E.drill_type, E.combine_position))
print("Cramer's V  drill_type x day      : %.3f" % cramers_v(E.drill_type, E.combine_day))
print("Cramer's V  position   x day      : %.3f" % cramers_v(E.combine_position, E.combine_day))
print()
print("position group x combine day (event counts):")
print(pd.crosstab(E.combine_position, E.combine_day).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
labels = ["drill_type", "drill_name", "position"]
vals = [eta_squared(E.order_frac, E.drill_type),
        eta_squared(E.order_frac, E.drill_name),
        eta_squared(E.order_frac, E.combine_position)]
axes[0].bar(labels, vals, color=["#1f3b73", "#4b7bb5", "#c0392b"])
axes[0].set_ylabel("eta^2  (share of within-session order variance explained)")
axes[0].set_title("Within-session ORDER ~ ...")
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.01, "%.3f" % v, ha="center")

ct = pd.crosstab(E.drill_type, E.combine_position)
axes[1].imshow(ct.values, aspect="auto", cmap="Blues")
axes[1].set_xticks(range(ct.shape[1])); axes[1].set_xticklabels(ct.columns, rotation=45, ha="right")
axes[1].set_yticks(range(ct.shape[0])); axes[1].set_yticklabels(ct.index, fontsize=8)
axes[1].set_title("drill_type x position group (event counts)")
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        axes[1].text(j, i, int(ct.values[i, j]), ha="center", va="center", fontsize=7)
plt.tight_layout()
plt.savefig(FIGDIR / "fig3_confounding.png", bbox_inches="tight")
plt.show()
print("saved", FIGDIR / "fig3_confounding.png")


**Q3 read:**
- **Within a session, order is driven by the fixed protocol**, not by the person: drill type explains
  ~**41%** of the variance in within-session order and drill *name* ~**85%** (the sequence is nearly
  deterministic), while **position explains ~0%** — everyone in a session runs the same sequence.
- **At the day level, position and drill type are heavily entangled**: Cramer's V(drill_type, position) ~
  **0.89**; each position group tests largely on its own day (position x day is nearly block-diagonal).
  So *between-session / between-day* variation cannot separate fatigue from position/ability.
- Consequence: a generic "later in the session = more tired" effect is **not identifiable** — "later"
  is almost the same thing as "different drill type". The **one clean handle** is repeated attempts
  *within a single drill* (drill type held fixed) — which Q2 shows carries little signal.

> **Verdict:** drill order is recoverable but **confounded with drill type**; the between-session
> axis is confounded with position group. Only *within-drill repeated attempts* give a clean fatigue
> contrast, and on this crude pass that contrast is ~null. See `../specs/de-risk-findings.md`.
